# Phase 2 - 04: 原生 Function Calling 验证

## 本节目标

在设计 tool-use 训练方案之前，先验证一个关键前提：
**Qwen3.5-2B 对原生 function calling 格式的支持程度如何？**

具体验证两件事：
1. 模型是否知道原生 tool_call 格式（chat template 如何渲染工具定义）
2. 面对需要计算的题目，模型是否会主动决定调用工具

## 背景

03 实验用的是自定义 `<expression>` 标签，相当于强制模型每题都必须写表达式。
这不是真正的 tool-use——真实场景是模型**自主判断何时需要工具**。

Qwen 系列模型支持原生 function calling，通过 `tokenizer.apply_chat_template(tools=...)` 传入工具定义，
模型会自动在 system prompt 里看到工具描述和调用格式，然后自主决定是否调用。

## 1. 导入依赖 & 加载模型

In [14]:
# 1-1 导入依赖 & 加载模型
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
MODEL_ID = os.path.expanduser("~/.cache/modelscope/hub/models/Qwen/Qwen3___5-2B")

print("加载 tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, local_files_only=True)
print("加载模型...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.float16, device_map=DEVICE, local_files_only=True
)
print(f"加载完成，设备: {next(model.parameters()).device}")

加载 tokenizer...
加载模型...


Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████████████████| 320/320 [00:04<00:00, 72.26it/s]

加载完成，设备: mps:0


## 2. 验证一：chat template 如何渲染工具定义

`tokenizer.apply_chat_template(tools=...)` 会自动把工具 JSON Schema 注入 system prompt，
并告知模型调用格式。先看渲染结果，了解模型"看到"的是什么。

In [16]:
# 2-1 查看 tools 渲染结果
TOOLS = [{
    "type": "function",
    "function": {
        "name": "calculator",
        "description": "计算数学表达式，返回精确结果",
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "合法的数学表达式，如 17 * 23"
                }
            },
            "required": ["expression"]
        }
    }
}]

messages = [{"role": "user", "content": "1847 × 293 等于多少？"}]
rendered = tokenizer.apply_chat_template(
    messages, tools=TOOLS, tokenize=False, add_generation_prompt=True
)
print(rendered)

<|im_start|>system
# Tools

You have access to the following functions:

<tools>
{"type": "function", "function": {"name": "calculator", "description": "计算数学表达式，返回精确结果", "parameters": {"type": "object", "properties": {"expression": {"type": "string", "description": "合法的数学表达式，如 17 * 23"}}, "required": ["expression"]}}}
</tools>

If you choose to call a function ONLY reply in the following format with NO suffix:

<tool_call>
<function=example_function_name>
<parameter=example_parameter_1>
value_1
</parameter>
<parameter=example_parameter_2>
This is the value for the second parameter
that can span
multiple lines
</parameter>
</function>
</tool_call>

<IMPORTANT>
Reminder:
- Function calls MUST follow the specified format: an inner <function=...></function> block must be nested within <tool_call></tool_call> XML tags
- Required parameters MUST be specified
- You may provide optional reasoning for your function call in natural language BEFORE the function call, but NOT after
- If there is n

## 3. 验证二：模型是否主动调用工具

用一组题目测试模型的决策行为：
- **简单题**：个位/两位数运算，模型应该直接答
- **难题**：4位数乘法，人算不出来，模型应该调工具

In [17]:
# 3-1 测试模型工具调用决策
def ask(question: str, max_new_tokens: int = 300) -> str:
    messages = [{"role": "user", "content": question}]
    text = tokenizer.apply_chat_template(
        messages, tools=TOOLS, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    new = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new, skip_special_tokens=True)


questions = [
    # 简单题（模型应直接答）
    ("简单", "3 + 5 等于多少？"),
    ("简单", "If I have 3 apples, how many do I have?"),
    # 难题（模型应调工具）
    ("难题", "1847 × 293 等于多少？"),
    ("难题", "一个工厂每天生产 1847 个零件，293 天能生产多少个？"),
    ("难题", "3929 * 4772 等于多少？"),
    ("难题", "What is 9876 multiplied by 5432?"),
]

for kind, q in questions:
    output = ask(q)
    used_tool = "<tool_call>" in output
    print(f"[{kind}] {q}")
    print(f"  调用工具: {'是 ✓' if used_tool else '否'}")
    print(f"  输出: {output.strip()}")
    print()

[简单] 3 + 5 等于多少？
  调用工具: 是 ✓
  输出: 3 + 5 等于 8。

<tool_call>
<function=calculator>
<parameter=expression>
3 + 5
</parameter>
</function>
</tool_call>

[简单] If I have 3 apples, how many do I have?
  调用工具: 否
  输出: If you have 3 apples, you have 3 apples.

[难题] 1847 × 293 等于多少？
  调用工具: 是 ✓
  输出: <tool_call>
<function=calculator>
<parameter=expression>
1847 * 293
</parameter>
</function>
</tool_call>

[难题] 一个工厂每天生产 1847 个零件，293 天能生产多少个？
  调用工具: 是 ✓
  输出: 要计算 293 天能生产多少个零件，我们可以将每天生产的数量乘以天数。

计算过程如下：
$$1847 \times 293$$

<tool_call>
<function=calculator>
<parameter=expression>
1847 * 293
</parameter>
</function>
</tool_call>

[难题] 3929 * 4772 等于多少？
  调用工具: 是 ✓
  输出: <tool_call>
<function=calculator>
<parameter=expression>
3929 * 4772
</parameter>
</function>
</tool_call>

[难题] What is 9876 multiplied by 5432?
  调用工具: 是 ✓
  输出: <tool_call>
<function=calculator>
<parameter=expression>
9876 * 5432
</parameter>
</function>
</tool_call>



## 4. 小结

### 实验结果

| 题目类型 | 调用工具 | 说明 |
|---|---|---|
| 简单加法 `3 + 5` | 否 | 直接回答，正确 |
| 非计算问题 | 否 | 直接回答，正确 |
| 大数乘法（直接问式） | **是** | 主动调 calculator |
| 大数乘法（自然语言） | **是** | 主动调 calculator |

### 结论

**Qwen3.5-2B 未经任何训练，已能正确做出工具调用决策：**
- 简单题 → 直接答，不调工具
- 大数计算 → 主动调 calculator，格式完全正确

这说明模型在 SFT 阶段已经学会了这个决策策略。
**针对这类计算场景做 RL 训练没有意义**——模型没有需要改进的地方。

### 对 tool-use RL 的启示

RL 能起作用的前提是**模型当前决策有缺陷**：
- 该用工具时不用（决策失误）
- 不该用工具时用了（过度调用）
- 调用格式错误（格式问题，但 SFT 已解决）

对于 Qwen3.5-2B + 计算器这个组合，三种缺陷都不存在。
要做有意义的 tool-use RL，需要找到模型**能力边界上的任务**：
模型自己处理成功率在 20-50% 左右，用工具后能到 90%+。